Proyek Klasifikasi Gambar: mikoajkolman/pokemon-images-first-generation17000-files

Nama: ASEP HARYANA SAPUTRA

Email: superaseph@gmail.com

ID Dicoding: mytheclipse

## 1. Persiapan Lingkungan

Mengimpor semua pustaka yang diperlukan dan mengatur seed untuk reproduksibilitas.

In [1]:
!pip install -r requirements.txt

In [2]:
# Import semua pustaka di awal
import os
import shutil
import zipfile
import random
import io
import logging
import traceback
import time
import psutil
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from PIL import Image
import splitfolders  # untuk split dataset

# Keras / TF
from tensorflow.keras.utils import image_dataset_from_directory
from tensorflow.keras import layers, models, callbacks, regularizers
from tensorflow.keras.applications import MobileNetV3Large
from tensorflow.keras.applications.mobilenet_v3 import preprocess_input
from tensorflow.keras.optimizers import AdamW,Adam

# Sklearn
from sklearn.metrics import confusion_matrix, classification_report

# Widget interaktif
import ipywidgets as widgets
from IPython.display import display, clear_output

# Konfigurasi global
IMG_SIZE = (256, 256)
BATCH_SIZE = 32
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Optimasi performa TF
os.environ["TF_DATA_AUTOTUNE_RAM_BUDGET"] = "4000000000"
os.environ['TF_XLA_FLAGS'] = '--tf_xla_enable_xla_devices=false'

# Gunakan mixed precision untuk mempercepat training (jika GPU mendukung)
tf.keras.mixed_precision.set_global_policy('mixed_float16')

print(f"Setup selesai. IMG_SIZE={IMG_SIZE}, BATCH_SIZE={BATCH_SIZE}")

2026-02-16 23:06:26.351999: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-16 23:06:26.360218: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771257986.370568  125233 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771257986.373970  125233 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1771257986.381792  125233 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

Setup selesai. IMG_SIZE=(256, 256), BATCH_SIZE=32


## 2. Ekstraksi Dataset

Dataset disediakan dalam file `pokemon.zip`. Ekstrak ke folder `dataset_raw`.

In [3]:
zip_filename = 'pokemon.zip'
extract_dir = 'dataset_raw'

# Ekstrak hanya jika folder belum ada atau kosong
if os.path.exists(extract_dir) and len(os.listdir(extract_dir)) > 0:
    print(f"Folder {extract_dir} sudah ada dan tidak kosong. Ekstraksi dilewati.")
else:
    if os.path.exists(extract_dir):
        shutil.rmtree(extract_dir)
    os.makedirs(extract_dir, exist_ok=True)
    with zipfile.ZipFile(zip_filename, 'r') as zip_ref:
        zip_ref.extractall(extract_dir)
    print("Ekstraksi selesai.")

Folder dataset_raw sudah ada dan tidak kosong. Ekstraksi dilewati.


## 3. Pembersihan Data

Langkah-langkah:
- Gabungkan kelas duplikat (misal 'Mr. Mime' dan 'MrMime').
- Hapus file non-gambar (selain .jpg, .jpeg, .png, .bmp).
- Hapus file gambar corrupt.
- Normalisasi metadata gambar (konversi ke RGB, hapus profil ICC).

In [4]:
# Tentukan direktori utama dataset
base_dir = os.path.join(extract_dir, 'pokemon')
if not os.path.exists(base_dir):
    base_dir = extract_dir  # jika tidak ada subfolder pokemon
print(f"Direktori utama: {base_dir}")

# 1. Gabungkan kelas duplikat
duplicate_map = {'Mr. Mime': 'MrMime'}
for src, dst in duplicate_map.items():
    src_path = os.path.join(base_dir, src)
    dst_path = os.path.join(base_dir, dst)
    if os.path.exists(src_path):
        if os.path.exists(dst_path):
            # Pindahkan semua file dari src ke dst
            for f in os.listdir(src_path):
                shutil.move(os.path.join(src_path, f), os.path.join(dst_path, f))
            shutil.rmtree(src_path)
            print(f"Digabungkan {src} ke {dst}")
        else:
            os.rename(src_path, dst_path)
            print(f"Diganti nama {src} -> {dst}")

# 2. Hapus file non-gambar dan corrupt
valid_ext = ('.jpg', '.jpeg', '.png', '.bmp')
removed = 0
for root, dirs, files in os.walk(base_dir):
    for file in files:
        fpath = os.path.join(root, file)
        ext = os.path.splitext(file)[1].lower()
        if ext not in valid_ext:
            os.remove(fpath)
            removed += 1
            continue
        try:
            with Image.open(fpath) as img:
                img.verify()  # cek integritas
        except Exception:
            os.remove(fpath)
            removed += 1
print(f"File dihapus (ekstensi tidak valid/corrupt): {removed}")

# 3. Bersihkan folder kelas yang kosong
for d in os.listdir(base_dir):
    dpath = os.path.join(base_dir, d)
    if os.path.isdir(dpath) and len(os.listdir(dpath)) == 0:
        shutil.rmtree(dpath)
        print(f"Folder kosong dihapus: {d}")

# 4. Normalisasi metadata (konversi ke RGB, hapus ICC profile)
sanitized = 0
for root, dirs, files in os.walk(base_dir):
    for file in files:
        fpath = os.path.join(root, file)
        try:
            with Image.open(fpath) as img:
                if img.mode != 'RGB' or 'icc_profile' in img.info:
                    img = img.convert('RGB')
                    img.save(fpath, 'JPEG', quality=95, optimize=True)
                    sanitized += 1
        except Exception:
            pass
print(f"Gambar dinormalisasi: {sanitized}")

# Hitung total gambar dan kelas
class_counts = {}
total_images = 0
for d in os.listdir(base_dir):
    dpath = os.path.join(base_dir, d)
    if os.path.isdir(dpath):
        cnt = len(os.listdir(dpath))
        class_counts[d] = cnt
        total_images += cnt
print(f"Total gambar valid: {total_images} dari {len(class_counts)} kelas")

Direktori utama: dataset_raw/pokemon
File dihapus (ekstensi tidak valid/corrupt): 0
Gambar dinormalisasi: 0
Total gambar valid: 16485 dari 142 kelas


## 4. Pembagian Dataset (Train:Validation:Test = 60:20:20)

Menggunakan library `split-folders` untuk membagi secara proporsional.

In [5]:
output_dir = 'dataset_split'
if os.path.exists(output_dir):
    shutil.rmtree(output_dir)

splitfolders.ratio(
    base_dir,
    output=output_dir,
    seed=SEED,
    ratio=(0.6, 0.2, 0.2),
    group_prefix=None
)

train_dir = os.path.join(output_dir, 'train')
val_dir = os.path.join(output_dir, 'val')
test_dir = os.path.join(output_dir, 'test')

def count_images(path):
    return sum(len(files) for _, _, files in os.walk(path))

print(f"Train set: {count_images(train_dir)} gambar")
print(f"Validation set: {count_images(val_dir)} gambar")
print(f"Test set: {count_images(test_dir)} gambar")

Copying files: 16485 files [00:00, 18289.99 files/s]

Train set: 9835 gambar
Validation set: 3239 gambar
Test set: 3411 gambar


## 5. Persiapan Data Loader dengan Augmentasi

Menggunakan `image_dataset_from_directory` untuk membaca gambar.  
Augmentasi diterapkan pada data training dengan intensitas sedang dan ditambah GaussianNoise.

In [ ]:
# Load dataset
train_ds = image_dataset_from_directory(
    train_dir,
    image_size=IMG_SIZE,
    batch_size=None,
    label_mode='categorical',
    shuffle=True,
    seed=SEED
)
class_names = train_ds.class_names
NUM_CLASSES = len(class_names)

# Batching
train_ds = train_ds.batch(BATCH_SIZE, drop_remainder=True)
val_ds = image_dataset_from_directory(
    val_dir, image_size=IMG_SIZE, batch_size=None, label_mode='categorical', shuffle=False, seed=SEED
).batch(BATCH_SIZE, drop_remainder=True)
test_ds = image_dataset_from_directory(
    test_dir, image_size=IMG_SIZE, batch_size=None, label_mode='categorical', shuffle=False, seed=SEED
).batch(BATCH_SIZE, drop_remainder=True)

print(f"Jumlah kelas: {NUM_CLASSES}")
print(f"Contoh kelas: {class_names[:5]}...")

# Augmentasi yang lebih bervariasi dengan penambahan GaussianNoise untuk regularisasi
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.2),
    layers.RandomContrast(0.15),
    layers.RandomTranslation(0.1, 0.1),
    layers.RandomBrightness(0.15),
    layers.GaussianNoise(0.01),  # Menambah noise kecil untuk regularisasi
])

# Fungsi preprocessing
def preprocess(images, labels):
    return preprocess_input(images), labels

def augment_and_preprocess(images, labels):
    x = data_augmentation(images)
    return preprocess_input(x), labels

# Terapkan
AUTOTUNE = tf.data.AUTOTUNE
train_generator = train_ds.map(augment_and_preprocess, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
validation_generator = val_ds.map(preprocess, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
test_generator = test_ds.map(preprocess, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)

Found 9835 files belonging to 142 classes.


I0000 00:00:1771257990.373840  125233 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 2998 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4050 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9


## 6. Visualisasi Sampel Data (dengan Augmentasi)

Menampilkan beberapa gambar dari training set yang telah diaugmentasi.

In [ ]:
sample_images, sample_labels = next(iter(train_generator))

plt.figure(figsize=(15, 8))
for i in range(15):
    plt.subplot(3, 5, i+1)
    img_np = sample_images[i].numpy().astype("uint8")
    plt.imshow(img_np)
    label_idx = np.argmax(sample_labels[i])
    plt.title(class_names[label_idx], fontsize=10)
    plt.axis('off')
plt.suptitle('Contoh Gambar Training Set (dengan Augmentasi)', fontsize=14)
plt.tight_layout()
plt.show()

## 7. Penanganan Ketidakseimbangan Kelas dengan Sample Weight

Hitung bobot untuk setiap kelas berdasarkan frekuensi di training set, lalu tambahkan ke pipeline data. Bobot dibatasi maksimum 3.0 untuk menghindari bobot ekstrem.

In [ ]:
# Hitung jumlah sampel per kelas di training
train_class_counts = Counter()
for class_name in class_names:
    folder = os.path.join(train_dir, class_name)
    if os.path.exists(folder):
        train_class_counts[class_name] = len(os.listdir(folder))

total_samples = sum(train_class_counts.values())
class_weights_list = []
for class_name in class_names:
    count = train_class_counts.get(class_name, 1)
    weight = total_samples / (NUM_CLASSES * count)
    # Batasi bobot maksimum agar tidak terlalu ekstrem
    weight = min(weight, 3.0)
    class_weights_list.append(weight)

class_weights_tensor = tf.constant(class_weights_list, dtype=tf.float32)

# Fungsi untuk menambahkan sample weight ke dataset
def add_sample_weight(image, label):
    class_indices = tf.argmax(label, axis=-1)
    sample_weights = tf.gather(class_weights_tensor, class_indices)
    return image, label, sample_weights

# Terapkan sample weight pada training generator
train_generator = train_generator.map(add_sample_weight, num_parallel_calls=AUTOTUNE)

# Tampilkan statistik bobot
print("Bobot kelas (dibatasi max 3.0):")
for i, name in enumerate(class_names[:10]):
    print(f"{name}: {class_weights_list[i]:.4f}")
print("...")

## 8. Membangun Model CNN dengan Transfer Learning

Menggunakan MobileNetV3Large sebagai basis dengan dropout internal diaktifkan (0.3). Ditambah layer klasifikasi dengan regularisasi L2 dan dropout lebih tinggi (0.6) untuk mengurangi overfitting.

In [ ]:
def create_model():
    base_model = MobileNetV3Large(
        input_shape=IMG_SIZE + (3,),
        include_top=False,
        weights='imagenet',
        include_preprocessing=False
    )
    base_model.trainable = False

    model = models.Sequential([
        layers.Input(shape=IMG_SIZE + (3,)),
        layers.Rescaling(1./255),
        base_model,
        
        layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
        layers.MaxPooling2D((2, 2)),
        
        layers.GlobalAveragePooling2D(),
        
        layers.Dense(
            128,
            activation='relu',
            kernel_regularizer=regularizers.l2(0.001)
        ),
        layers.BatchNormalization(),
        layers.Dropout(0.2),
        
        layers.Dense(NUM_CLASSES, activation='softmax')
    ])
    return model

checkpoint_path = 'submission/best_model.keras'
if os.path.exists(checkpoint_path):
    try:
        model = models.load_model(checkpoint_path)
    except:
        model = create_model()
else:
    model = create_model()

model.summary()

## 9. Callbacks

Menggunakan ModelCheckpoint, EarlyStopping (patience 20), ReduceLROnPlateau, CSVLogger, dan TensorBoard.

In [ ]:
# Direktori untuk logs
log_dir = "logs/fit/" + time.strftime("%Y%m%d-%H%M%S")
os.makedirs("submission", exist_ok=True)

checkpoint = callbacks.ModelCheckpoint(
    'submission/best_model.keras',
    monitor='val_accuracy',
    save_best_only=True,
    mode='max',
    verbose=1
)

early_stop = callbacks.EarlyStopping(
    monitor='val_accuracy',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.2,
    patience=3,
    min_lr=1e-7,
    verbose=1
)

csv_logger = callbacks.CSVLogger('submission/training_log.csv')
tensorboard = callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1)

callbacks_list = [checkpoint, early_stop, reduce_lr, csv_logger, tensorboard]

## 10. Pelatihan Model (Fase 1: Head Saja)

Melatih layer klasifikasi terlebih dahulu dengan base model dibekukan. Menggunakan optimizer AdamW dengan weight decay dan label smoothing 0.15.

In [ ]:
model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history1 = model.fit(
    train_generator,
    epochs=40,
    validation_data=validation_generator,
    callbacks=callbacks_list
)

## 11. Fine-Tuning (Fase 2: Semua Layer)

Membuka semua layer base model dan melanjutkan pelatihan dengan learning rate sangat kecil (5e-6).

In [ ]:
base_model = model.layers[1]
base_model.trainable = True

for layer in base_model.layers[:-50]:
    layer.trainable = False

model.compile(
    optimizer=AdamW(learning_rate=5e-6, weight_decay=1e-4), 
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.15),
    metrics=['accuracy']
)

history_fine = model.fit(
    train_generator,
    epochs=150,
    initial_epoch=history1.epoch[-1] + 1,
    validation_data=validation_generator,
    callbacks=callbacks_list
)

## 12. Plot Akurasi dan Loss

Menggabungkan riwayat dari kedua fase pelatihan.

In [ ]:
acc = history1.history['accuracy'] + history_fine.history['accuracy']
val_acc = history1.history['val_accuracy'] + history_fine.history['val_accuracy']
loss = history1.history['loss'] + history_fine.history['loss']
val_loss = history1.history['val_loss'] + history_fine.history['val_loss']

plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.plot(acc, label='Training Accuracy')
plt.plot(val_acc, label='Validation Accuracy')
plt.axvline(x=len(history1.history['accuracy'])-1, color='gray', linestyle='--', label='Fine-tuning start')
plt.legend()
plt.title('Akurasi Training dan Validasi')

plt.subplot(1, 2, 2)
plt.plot(loss, label='Training Loss')
plt.plot(val_loss, label='Validation Loss')
plt.axvline(x=len(history1.history['loss'])-1, color='gray', linestyle='--', label='Fine-tuning start')
plt.legend()
plt.title('Loss Training dan Validasi')
plt.show()

## 13. Evaluasi pada Test Set

Menggunakan model terbaik yang disimpan oleh ModelCheckpoint.

In [ ]:
best_model = tf.keras.models.load_model('submission/best_model.keras')

test_loss, test_accuracy = best_model.evaluate(test_generator, verbose=1)
print(f"\nTest Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

if test_accuracy >= 0.95:
    print("✅ Model memenuhi target akurasi >=95%.")
else:
    print("⚠️  Model belum mencapai 95%.")

## 14. Confusion Matrix dan Classification Report

Menampilkan matriks kebingungan dan laporan klasifikasi per kelas.

In [ ]:
# Kumpulkan label sebenarnya dan prediksi
y_true = []
y_pred_probs = []
for images, labels in test_generator:
    y_true.extend(np.argmax(labels.numpy(), axis=1))
    preds = best_model.predict(images, verbose=0)
    y_pred_probs.extend(preds)

y_true = np.array(y_true)
y_pred = np.argmax(np.array(y_pred_probs), axis=1)

# Laporan klasifikasi
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=class_names))

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(20, 20))
plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
plt.title('Confusion Matrix')
plt.colorbar()
tick_marks = np.arange(len(class_names))
plt.xticks(tick_marks, class_names, rotation=90, fontsize=8)
plt.yticks(tick_marks, class_names, fontsize=8)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.show()

## 15. Simpan Model dalam Berbagai Format

Menyimpan model dalam format SavedModel, TF-Lite, H5 (keras), dan TFJS.

In [ ]:
%run export_model.py

## 16. Inference dengan Model TF-Lite

Menguji model TF-Lite pada sampel acak dari test set.

In [ ]:
# Load interpreter
interpreter = tf.lite.Interpreter(model_path='submission/tflite/model.tflite')
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

input_index = input_details[0]['index']
input_dtype = input_details[0]['dtype']
input_scale, input_zero_point = input_details[0]['quantization']

output_index = output_details[0]['index']
output_scale, output_zero_point = output_details[0]['quantization']

# Ambil batch acak dari test set
viz_ds = image_dataset_from_directory(
    test_dir,
    image_size=IMG_SIZE,
    batch_size=32,
    label_mode='categorical',
    shuffle=True,
    seed=random.randint(0, 1000)
)

images, labels = next(iter(viz_ds.map(preprocess)))

plt.figure(figsize=(20, 10))

for i in range(15):
    img = images[i].numpy()
    true_idx = np.argmax(labels[i])
    true_name = class_names[true_idx]

    # Persiapan input
    input_data = np.expand_dims(img, axis=0)

    if input_dtype == np.uint8 or input_dtype == np.int8:
        # Quantized model
        input_data = input_data / input_scale + input_zero_point
        input_data = np.round(input_data).astype(input_dtype)
    else:
        # Float model
        input_data = input_data.astype(input_dtype)

    interpreter.set_tensor(input_index, input_data)
    interpreter.invoke()

    output = interpreter.get_tensor(output_index)

    # Dequantize jika perlu
    if output_scale > 0:
        output = (output.astype(np.float32) - output_zero_point) * output_scale

    pred_idx = np.argmax(output[0])
    pred_name = class_names[pred_idx]
    conf = float(output[0][pred_idx]) * 100

    display_img = img.astype("uint8")

    plt.subplot(3, 5, i + 1)
    plt.imshow(display_img)
    color = 'green' if pred_idx == true_idx else 'red'
    plt.title(f"True: {true_name}\nPred: {pred_name}\n({conf:.1f}%)",
              fontsize=8, color=color)
    plt.axis('off')

plt.suptitle("Inferensi TF-Lite pada 15 Sampel Acak")
plt.tight_layout()
plt.show()

## 17. Inference Interaktif (Upload Gambar) dengan Grad-CAM

Widget untuk mengunggah gambar sendiri dan melihat prediksi beserta Grad-CAM.
Layer konvolusi terakhir dicari secara otomatis dari base model.

In [ ]:
# Fungsi Grad-CAM yang robust
def find_last_conv_layer(model):
    """Cari layer konvolusi terakhir dalam base model."""
    base_model = model.layers[0]
    for layer in reversed(base_model.layers):
        if isinstance(layer, tf.keras.layers.Conv2D):
            return layer.name
    raise ValueError("Tidak ditemukan layer Conv2D dalam base model.")

def make_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index=None):
    base_model = model.layers[0]
    
    # Model yang mengeluarkan output dari conv layer dan base model
    grad_model = tf.keras.models.Model(
        inputs=[base_model.inputs],
        outputs=[base_model.get_layer(last_conv_layer_name).output, base_model.output]
    )
    
    with tf.GradientTape() as tape:
        conv_output, base_output = grad_model(img_array)
        # Forward melalui classifier head
        x = base_output
        for layer in model.layers[1:]:
            x = layer(x, training=False)
        preds = x
        if pred_index is None:
            pred_index = tf.argmax(preds[0])
        class_channel = preds[:, pred_index]

    grads = tape.gradient(class_channel, conv_output)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_output = conv_output[0]
    heatmap = conv_output @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)
    return heatmap.numpy()

def save_and_display_gradcam(img, heatmap, alpha=0.4):
    img = tf.keras.preprocessing.image.img_to_array(img)
    heatmap = np.uint8(255 * heatmap)
    jet = plt.colormaps.get_cmap("jet")
    jet_colors = jet(np.arange(256))[:, :3]
    jet_heatmap = jet_colors[heatmap]
    jet_heatmap = tf.keras.preprocessing.image.array_to_img(jet_heatmap)
    jet_heatmap = jet_heatmap.resize((img.shape[1], img.shape[0]))
    jet_heatmap = tf.keras.preprocessing.image.img_to_array(jet_heatmap)
    superimposed_img = jet_heatmap * alpha + img
    return tf.keras.preprocessing.image.array_to_img(superimposed_img)

# Widget upload
upload_btn = widgets.FileUpload(accept='image/*', multiple=False, description='Upload', icon='upload')
out = widgets.Output()

last_conv_name = find_last_conv_layer(best_model)
print(f"Layer konvolusi terakhir: {last_conv_name}")

def on_upload(change):
    # Handle berbagai versi ipywidgets
    if isinstance(upload_btn.value, tuple):
        uploaded_file = upload_btn.value[0]
    else:
        uploaded_file = next(iter(upload_btn.value.values()))
    content = uploaded_file['content']
    try:
        image = Image.open(io.BytesIO(content)).convert('RGB')
        with out:
            clear_output(wait=True)
            plt.figure(figsize=(10, 5))
            plt.subplot(1, 2, 1)
            plt.imshow(image)
            plt.title("Gambar Asli")
            plt.axis('off')

            # Preprocessing
            img_resized = image.resize(IMG_SIZE)
            img_array = tf.keras.preprocessing.image.img_to_array(img_resized)
            img_array = tf.expand_dims(img_array, 0)
            img_array = preprocess_input(img_array)

            # Prediksi
            preds = best_model.predict(img_array, verbose=0)
            pred_idx = np.argmax(preds[0])
            pred_label = class_names[pred_idx]
            confidence = preds[0][pred_idx]

            # Grad-CAM
            heatmap = make_gradcam_heatmap(img_array, best_model, last_conv_name, pred_idx)
            result = save_and_display_gradcam(img_resized, heatmap)

            plt.subplot(1, 2, 2)
            plt.imshow(result)
            plt.title(f"Pred: {pred_label} ({confidence:.2%})")
            plt.axis('off')
            plt.show()
    except Exception as e:
        with out:
            clear_output()
            print(f"Error: {e}")
            traceback.print_exc()

upload_btn.observe(on_upload, names='value')
display(widgets.VBox([widgets.Label("Upload Gambar Pokemon:"), upload_btn, out]))


## 18. Model Card (Dokumentasi)

Informasi lengkap tentang model yang telah dilatih.

- **Nama Model**: Pokemon-Gen1-MobileNetV3-Large (v2.1 - Peningkatan Regularisasi)
- **Versi**: 2.1
- **Arsitektur**: MobileNetV3Large (Transfer Learning) dengan dropout internal 0.3 + classifier Dense 1024 (L2 reg) + Dropout 0.6
- **Framework**: TensorFlow 2.x / Keras
- **Dataset**: ~17.000 gambar Pokemon Generasi 1 (140+ kelas)
- **Target Akurasi**: ≥95% pada validasi dan test
- **Input**: Gambar RGB 256x256
- **Output**: Probabilitas untuk setiap kelas Pokemon
- **Cara Pakai**: Gunakan model untuk klasifikasi gambar Pokemon Generasi 1.
- **Etika**: Model ini hanya untuk tujuan edukasi. Hak cipta Pokemon milik Nintendo/Game Freak.

### Ringkasan Perubahan untuk Mencapai >95%
1. **Regularisasi lebih kuat**:
   - L2 regularization (1e-4) pada Dense layer.
   - Dropout ditingkatkan menjadi 0.6.
   - Dropout internal MobileNet diaktifkan (0.3).
2. **Augmentasi ditambah GaussianNoise** untuk variasi data.
3. **Label smoothing ditingkatkan menjadi 0.15**.
4. **Pembatasan class weight lebih ketat** (max 3.0).
5. **Fine-tuning dengan learning rate lebih kecil** (5e-6).
6. **Early stopping patience dinaikkan** menjadi 20.
7. **Optimizer AdamW tetap digunakan** dengan weight decay.
8. **Evaluasi dan monitoring** dengan TensorBoard dan CSVLogger.